<div align="left">

<h2><b>Training the model for other variables. </b></h2>

</div>

#### <u><strong> Setup.</strong></u>

In [37]:
# Needed packages
import pandas as pd
import numpy as np
import os
import zipfile
import tensorflow as tf
from tensorflow import keras
from google.colab import drive
import math

# Mount drive and set current directory
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/London School of Economics/Deep Learning/DeepLearningProject/Code')

# Needed directories
PHOTO_DIR  = "../Input/GoogleStreetViewImagesNew.zip"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### <u><strong> Precursor Functions.</strong></u>

In [ ]:
##############################
# I. Data loading (Modified) #
##############################
# Sources:
# Assignment I dataloading fn &
# https://medium.com/@kumudtraveldiaries/step-by-step-preprocessing-guide-for-images-in-both-cnn-and-dense-layer-pipelines-1994c3ad3e87
def dataloading(zip_path=PHOTO_DIR,
                img_size=128,
                batch_size=32,
                preprocess_fn=None,
                variable="income_group"):

    # 0. CSV Definitions
    if variable == "income_group":
      csv_path="../Output/images_bra.csv"
    elif variable == "shr_college_group":
      csv_path="../Output/images_bra_shr_college.csv"
    elif variable == "vacancy_rate_group":
      csv_path="../Output/images_bra_vacancy.csv"
    elif variable == "median_home_value_group":
      csv_path="../Output/images_bra_home_value.csv"

    # 1. Load CSV
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[variable]).reset_index(drop=True)

    # 2. Build group IDs
    df["loc_id"] = df["file_path"].apply(
        lambda f: os.path.basename(f).split("_h")[0]
    )

    # 3. Train / Val / Test split
    locs = df["loc_id"].unique()
    np.random.seed(50)
    np.random.shuffle(locs)
    n_test = int(len(locs) * 0.15)
    n_val = int(len(locs) * 0.15)
    test_locs = locs[:n_test]
    val_locs = locs[n_test:n_test+n_val]
    train_locs = locs[n_test+n_val:]
    train_df = df[df["loc_id"].isin(train_locs)]
    val_df   = df[df["loc_id"].isin(val_locs)]
    test_df  = df[df["loc_id"].isin(test_locs)]
    train_labels = train_df[variable].values.astype(np.int64)
    val_labels   = val_df[variable].values.astype(np.int64)
    test_labels  = test_df[variable].values.astype(np.int64)
    splits = {
        "train": (train_df["file_path"].values, train_labels),
        "val":   (val_df["file_path"].values, val_labels),
        "test":  (test_df["file_path"].values, test_labels),
    }
    for k, (p, l) in splits.items():
        print(f"{k}: {len(p)} images")

    # 4. Open zip once
    zip_ref = zipfile.ZipFile(zip_path, "r")

    # 5. Image loading function
    def load_image_from_zip(path, label):
        def _read(path_tensor):
            p = path_tensor.numpy().decode("utf-8")
            img_data = zip_ref.read(p)
            img = tf.image.decode_jpeg(img_data, channels=3)
            img = tf.image.resize(img, [img_size, img_size])
            img = tf.cast(img, tf.float32)
            if preprocess_fn:
                img = preprocess_fn(img)
            else:
                img = img / 255.0
            return img
        img = tf.py_function(_read, [path], tf.float32)
        img.set_shape([img_size, img_size, 3])
        return img, label

    # 6. Build tf.data pipelines
    def make_dataset(file_paths, lbls, is_train=False):
        ds = tf.data.Dataset.from_tensor_slices((file_paths, lbls))
        if is_train:
            ds = ds.shuffle(len(file_paths))
        ds = ds.map(load_image_from_zip, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

    train_ds = make_dataset(*splits["train"], is_train=True)
    val_ds   = make_dataset(*splits["val"])
    test_ds  = make_dataset(*splits["test"])

    return train_ds, val_ds, test_ds, train_df, val_df, test_df


def dataloading_multiview(variable, img_size=384, batch_size=16, preprocess_fn=None):

    # CSV Definitions
    if variable == "income_group":
      csv_path="../Output/images_bra.csv"
    elif variable == "shr_college_group":
      csv_path="../Output/images_bra_shr_college.csv"
    elif variable == "vacancy_rate_group":
      csv_path="../Output/images_bra_vacancy.csv"
    elif variable == "median_home_value_group":
      csv_path="../Output/images_bra_home_value.csv"

    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[variable]).reset_index(drop=True)
    df["full_path"] = df["file_path"].apply(lambda f: os.path.join("../Input/", f))
    df["loc_id"] = df["file_path"].apply(lambda f: os.path.basename(f).split("_h")[0])
    df["heading"] = df["file_path"].apply(lambda f: int(os.path.basename(f).split("_h")[1].replace(".jpg", "")))
    df = df[df["full_path"].apply(os.path.exists)].reset_index(drop=True)

    grouped = (df.sort_values("heading").groupby("loc_id").agg(paths=("full_path", list), label=(variable, "first")).reset_index())
    grouped = grouped[grouped["paths"].apply(len) == 4].reset_index(drop=True)
    print(f"Total locations with all 4 headings: {len(grouped)} "
          f"({len(grouped)*4} images)")

    np.random.seed(50)
    loc_ids = grouped["loc_id"].values.copy()
    np.random.shuffle(loc_ids)
    n = len(loc_ids)
    train_ids = set(loc_ids[:int(0.70 * n)])
    val_ids   = set(loc_ids[int(0.70 * n):int(0.85 * n)])
    test_ids  = set(loc_ids[int(0.85 * n):])

    train_df = grouped[grouped["loc_id"].isin(train_ids)].reset_index(drop=True)
    val_df   = grouped[grouped["loc_id"].isin(val_ids)].reset_index(drop=True)
    test_df  = grouped[grouped["loc_id"].isin(test_ids)].reset_index(drop=True)

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} locations")

    def make_dataset(split_df, shuffle=False):
        paths_list = split_df["paths"].tolist()
        labels     = split_df["label"].values.astype(np.int32)

        def generator():
            for paths, label in zip(paths_list, labels):
                yield paths, label

        ds = tf.data.Dataset.from_generator(
            generator,
            output_signature=(
                tf.TensorSpec(shape=(4,), dtype=tf.string),
                tf.TensorSpec(shape=(),   dtype=tf.int32)
            )
        )

        def load_fn(paths, label):
            imgs = tf.map_fn(
                lambda p: preprocess_fn(
                    tf.cast(
                        tf.image.resize(
                            tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
                            [img_size, img_size]
                        ), tf.float32
                    )
                ),
                paths,
                fn_output_signature=tf.TensorSpec(
                    shape=(img_size, img_size, 3), dtype=tf.float32
                )
            )
            return imgs, label

        if shuffle:
            ds = ds.shuffle(buffer_size=len(split_df), seed=42)
        ds = (ds
              .map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(batch_size)
              .repeat()
              .prefetch(tf.data.AUTOTUNE))
        return ds

    train_ds = make_dataset(train_df, shuffle=True)
    val_ds   = make_dataset(val_df)
    test_ds  = make_dataset(test_df)

    steps = {
        "train": math.ceil(len(train_df) / batch_size),
        "val":   math.ceil(len(val_df)   / batch_size),
        "test":  math.ceil(len(test_df)  / batch_size),
    }
    print(f"Steps — train: {steps['train']} | val: {steps['val']} | test: {steps['test']}")

    return train_ds, val_ds, test_ds, steps, train_df, val_df, test_df

#####################################
# II. Model + aggregation + Running #
#####################################

# Run model fn
def run_model(model_path, train_ds, val_ds, test_ds):
  # Augmentation
  data_augmentation = tf.keras.Sequential([
      tf.keras.layers.RandomFlip("horizontal"),
      tf.keras.layers.RandomRotation(0.15),
      tf.keras.layers.RandomZoom(0.15),
      tf.keras.layers.RandomBrightness(0.2),
      tf.keras.layers.RandomContrast(0.2),
      tf.keras.layers.RandomTranslation(0.1, 0.1),
  ], name="data_augmentation")

  # EfficientNetV2S backbone
  base_model = tf.keras.applications.EfficientNetV2S(
      input_shape=(384, 384, 3),
      include_top=False,
      weights='imagenet'
  )
  base_model.trainable = False

  # Build model
  inputs = tf.keras.Input(shape=(384, 384, 3), name="input_images")
  x = data_augmentation(inputs)

  x = base_model(x, training=False)

  x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
  x = tf.keras.layers.Dropout(0.4, name="dropout_1")(x)
  x = tf.keras.layers.Dense(
          512, activation="relu",
          kernel_regularizer=tf.keras.regularizers.l2(1e-4),
          name="dense_hidden"
      )(x)
  x = tf.keras.layers.Dropout(0.4, name="dropout_2")(x)
  outputs = tf.keras.layers.Dense(
              3, activation="softmax",
              kernel_regularizer=tf.keras.regularizers.l2(1e-4),
              name="predictions"
            )(x)

  model = tf.keras.Model(inputs, outputs, name="EfficientNetV2S_Improved")
  model.summary()

  # Feature extraction
  print("\n=== Feature Extraction ===")

  model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
      loss=tf.keras.losses.SparseCategoricalCrossentropy(),
      metrics=['accuracy']
  )

  history_p1 = model.fit(
      train_ds,
      validation_data=val_ds,
      epochs=20,
      verbose=1,
      callbacks=[
          tf.keras.callbacks.EarlyStopping(
              monitor='val_accuracy', patience=10,
              restore_best_weights=True, verbose=1
          ),
          tf.keras.callbacks.ModelCheckpoint(
              filepath=model_path, monitor='val_accuracy',
              save_best_only=True, verbose=1
          ),
          tf.keras.callbacks.ReduceLROnPlateau(
              monitor='val_loss', factor=0.2, patience=4,
              min_lr=1e-6, verbose=1
          ),
              ]
  )

  print("\n=== Test Evaluation (Phase 1) ===")
  loss_p1, acc_p1 = model.evaluate(test_ds, verbose=1)
  print(f"Test accuracy : {acc_p1:.4f}  |  Test loss : {loss_p1:.4f}")

  # Fine-tuning
  print("\n=== Fine-Tuning ===")

  base_model.trainable = True

  fine_tune_at = 340
  for layer in base_model.layers[:fine_tune_at]:
      layer.trainable = False

  for layer in base_model.layers:
      if isinstance(layer, tf.keras.layers.BatchNormalization):
          layer.trainable = False

  trainable_count = sum(1 for l in base_model.layers if l.trainable)
  print(f"Trainable backbone layers: {trainable_count} / {len(base_model.layers)}")

  total_steps = 30 * len(train_ds)
  cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
      initial_learning_rate=1e-5,
      decay_steps=total_steps,
      alpha=1e-7,
  )

  model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_lr),
      loss=tf.keras.losses.SparseCategoricalCrossentropy(),
      metrics=['accuracy']
  )

  history_p2 = model.fit(
      train_ds,
      validation_data=val_ds,
      epochs=30,
      callbacks=[
      tf.keras.callbacks.EarlyStopping(
          monitor='val_accuracy', patience=10,
          restore_best_weights=True, verbose=1
      ),
      tf.keras.callbacks.ModelCheckpoint(
          filepath=model_path,
          monitor='val_accuracy', save_best_only=True, verbose=1
      ),
                ],
      verbose=1
  )

  print("Test Evaluation (Phase 2 — Fine-tuned)")
  loss_p2, acc_p2 = model.evaluate(test_ds, verbose=1)
  print(f"Test accuracy : {acc_p2:.4f}  |  Test loss : {loss_p2:.4f}")

# Load model and aggregate fn
def load_model_and_aggregate(model_path, train_ds, test_ds, test_df):
    # Load model
    model = keras.models.load_model(model_path, compile=False)

    total_steps = 30 * len(train_ds)

    cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-5,
        decay_steps=total_steps,
        alpha=1e-7,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    # Image-level sanity check
    loss, acc = model.evaluate(test_ds, verbose=1)
    print(f"\nImage-level sanity check: Loss: {loss:.4f} | Accuracy: {acc:.4f}")

    # Predict once
    all_probs = []
    all_labels = []

    for x_batch, y_batch in test_ds:
        probs = model.predict(x_batch, verbose=0)
        all_probs.append(probs)
        all_labels.append(y_batch.numpy())

    probs = np.concatenate(all_probs)
    true = np.concatenate(all_labels)

    # Image-level results
    results_img = test_df.copy().reset_index(drop=True)
    results_img["actual"] = true

    for c in range(probs.shape[1]):
        results_img[f"prob_{c}"] = probs[:, c]

    prob_cols = [f"prob_{c}" for c in range(probs.shape[1])]

    # Median aggregation by loc_id
    results = (
        results_img
        .groupby("loc_id")
        .agg(
            actual=("actual", "first"),
            GEOID=("GEOID", "first"),
            file_path=("file_path", "first"),
            **{col: (col, "median") for col in prob_cols}
        )
        .reset_index()
    )

    results["predicted"] = results[prob_cols].values.argmax(axis=1)

    # Aggregated accuracy
    agg_acc = (results["actual"] == results["predicted"]).mean()
    print(f"Median aggregation accuracy: {agg_acc:.4f}")

    return model, results

def run_whole(variable):

  # Load data
    train_ds, val_ds, test_ds, train_df, val_df, test_df = dataloading(
        img_size=384,
        batch_size=16,
        preprocess_fn=tf.keras.applications.efficientnet_v2.preprocess_input,
        variable=variable)

  # Run model
    model_path=f"../Results/best_efficientnetv2s_{variable}.keras"
    run_model(model_path=model_path, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds)
    load_model_and_aggregate(model_path=model_path, train_ds=train_ds, test_ds=test_ds, test_df=test_df)

def run_LSTM(variable, IMG_SIZE = 384, BATCH_SIZE = 16):
  train_ds, val_ds, test_ds, steps, train_df, val_df, test_df = dataloading_multiview(
                  variable=variable,
                  img_size=IMG_SIZE,
                  batch_size=BATCH_SIZE,
                  preprocess_fn=tf.keras.applications.efficientnet_v2.preprocess_input
                  )

# Verify shapes
  image_batch, label_batch = next(iter(train_ds))
  print("Image batch shape :", image_batch.shape) # (16, 4, 384, 384, 3)
  print("Label batch shape :", label_batch.shape) # (16,)

  IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

# Augmentation
# Applied per-view via TimeDistributed — each view gets its own random augment
  data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal"),
  tf.keras.layers.RandomRotation(0.15),
  tf.keras.layers.RandomZoom(0.15),
  tf.keras.layers.RandomBrightness(0.2),
  tf.keras.layers.RandomContrast(0.2),
  tf.keras.layers.RandomTranslation(0.1, 0.1),
  ], name="data_augmentation")

# Shared EfficientNetV2S backbone
  base_model = tf.keras.applications.EfficientNetV2S(
                                  input_shape=IMG_SHAPE,
                                  include_top=False,
                                  weights='imagenet'
                                  )
  base_model.trainable = False

  gap = tf.keras.layers.GlobalAveragePooling2D(name="gap")

# Model with LSTM fusion
  inputs = tf.keras.Input(shape=(4, IMG_SIZE, IMG_SIZE, 3), name="multiview_input")

  x = tf.keras.layers.TimeDistributed(
  data_augmentation, name="td_augmentation"
  )(inputs)

  x = tf.keras.layers.TimeDistributed(
  base_model, name="td_backbone"
  )(x, training=False) # (batch, 4, 12, 12, 1280)

  x = tf.keras.layers.TimeDistributed(
  gap, name="td_gap"
  )(x) # (batch, 4, 1280)

# LSTM fusion
  x = tf.keras.layers.LSTM(
  256,
  return_sequences=False,
  dropout=0.2, # input dropout within LSTM
  recurrent_dropout=0.0, # keep recurrent path clean for stability
  name="view_lstm"
  )(x) # (batch, 256)

# Head
  x = tf.keras.layers.Dropout(0.4, name="dropout_1")(x)
  x = tf.keras.layers.Dense(
  512, activation="relu",
  kernel_regularizer=tf.keras.regularizers.l2(1e-4),
  name="dense_hidden"
  )(x)
  x = tf.keras.layers.Dropout(0.3, name="dropout_2")(x)
  outputs = tf.keras.layers.Dense(
  3, activation="softmax",
  kernel_regularizer=tf.keras.regularizers.l2(1e-4),
  name="predictions"
  )(x)

  model = tf.keras.Model(inputs, outputs, name="EfficientNetV2S_LSTM_MultiView")
  model.summary()

# Callbacks
  def make_callbacks(ckpt="best_efficientnetv2s_lstm.keras"):
    return [
    tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=10,
    restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
    filepath=ckpt, monitor='val_accuracy',
    save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=4,
    min_lr=1e-6, verbose=1
    ),
    ]

# Phase 1 — Feature extraction (backbone frozen)
  print("\n=== Phase 1: Feature Extraction (frozen backbone) ===")

  model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
  )

  history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    steps_per_epoch=steps["train"],
    validation_steps=steps["val"],
    callbacks=make_callbacks(),
    verbose=1
  )

  print("\n=== Test Evaluation (Phase 1) ===")
  loss_p1, acc_p1 = model.evaluate(test_ds, steps=steps["test"], verbose=1)
  print(f"Test accuracy : {acc_p1:.4f} | Test loss : {loss_p1:.4f}")

# Phase 2 — Fine-tuning
  print("\n=== Phase 2: Fine-Tuning (top 30% of backbone unfrozen) ===")

  base_model.trainable = True

  fine_tune_at = int(len(base_model.layers) * 0.70) # freeze bottom 70%
  for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Keep all BatchNorm layers frozen — critical for EfficientNet with small batches
  for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
      layer.trainable = False

  trainable_count = sum(1 for l in base_model.layers if l.trainable)
  print(f"Freezing layers 0–{fine_tune_at} / {len(base_model.layers)}")
  print(f"Trainable backbone layers: {trainable_count} / {len(base_model.layers)}")

  total_steps = 30 * steps["train"]
  cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
  initial_learning_rate=1e-5,
  decay_steps=total_steps,
  alpha=1e-7,
  )

  model.compile(
  optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_lr),
  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
  metrics=['accuracy']
  )

  callbacks_p2 = [
  tf.keras.callbacks.EarlyStopping(
  monitor='val_accuracy', patience=10,
  restore_best_weights=True, verbose=1
  ),
  tf.keras.callbacks.ModelCheckpoint(
  filepath="best_efficientnetv2s_lstm.keras",
  monitor='val_accuracy', save_best_only=True, verbose=1
  ),
  ]

  history_p2 = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=30,
  steps_per_epoch=steps["train"],
  validation_steps=steps["val"],
  callbacks=callbacks_p2,
  verbose=1
  )

  print("\n=== Test Evaluation (Phase 2 — Fine-tuned) ===")
  loss_p2, acc_p2 = model.evaluate(test_ds, steps=steps["test"], verbose=1)
  print(f"Test accuracy : {acc_p2:.4f} | Test loss : {loss_p2:.4f}")
  print(f"Single-view EfficientNetV2S baseline : 66.35%")
  print(f"LSTM multi-view (Phase 2) : {acc_p2*100:.2f}%")

<div align="left">

<h2><b><u>Model I: Median Aggregator.</u></b></h2>

</div>

#### <u><strong> Eduction Group.</strong></u>

In [ ]:
run_whole(variable="shr_college_group")

train: 9756 images
val: 2084 images
test: 2088 images


Model: "EfficientNetV2S_Improved"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_images (InputLayer)       │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 12, 12, 1280)   │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,988,771 (80.07 MB)

 Trainable params: 657,411 (2.51 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


=== Feature Extraction ===
Epoch 1/20
610/610 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.4095 - loss: 1.2086
Epoch 1: val_accuracy improved from None to 0.46305, saving model to ../Results/best_efficientnetv2s_shr_college_group.keras

Epoch 1: finished saving model to ../Results/best_efficientnetv2s_shr_college_group.keras
610/610 ━━━━━━━━━━━━━━━━━━━━ 77s 93ms/step - accuracy: 0.4267 - loss: 1.1592 - val_accuracy: 0.4631 - val_loss: 1.0976 - learning_rate: 0.0010
Epoch 2/20
609/610 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.4679 - loss: 1.1032
Epoch 2: val_accuracy improved from 0.46305 to 0.48656, saving model to ../Results/best_efficientnetv2s_shr_college_group.keras

Epoch 2: finished saving model to ../Results/best_efficientnetv2s_shr_college_group.keras
610/610 ━━━━━━━━━━━━━━━━━━━━ 50s 82ms/step - accuracy: 0.4588 - loss: 1.1079 - val_accuracy: 0.4866 - val_loss: 1.0671 - learning_rate: 0.0010
Epoch 3/20
609/610 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.4747 - los

#### <u><strong> Vacancy Group.</strong></u>

In [ ]:
run_whole(variable="vacancy_rate_group")

train: 12900 images
val: 2764 images
test: 2764 images


Model: "EfficientNetV2S_Improved"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_images (InputLayer)       │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 12, 12, 1280)   │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,988,771 (80.07 MB)

 Trainable params: 657,411 (2.51 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


=== Feature Extraction ===
Epoch 1/20
807/807 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4463 - loss: 1.1528
Epoch 1: val_accuracy improved from None to 0.56946, saving model to ../Results/best_efficientnetv2s_vacancy_rate_group.keras

Epoch 1: finished saving model to ../Results/best_efficientnetv2s_vacancy_rate_group.keras
807/807 ━━━━━━━━━━━━━━━━━━━━ 102s 101ms/step - accuracy: 0.4629 - loss: 1.1082 - val_accuracy: 0.5695 - val_loss: 0.9895 - learning_rate: 0.0010
Epoch 2/20
806/807 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4862 - loss: 1.0620
Epoch 2: val_accuracy did not improve from 0.56946
807/807 ━━━━━━━━━━━━━━━━━━━━ 65s 80ms/step - accuracy: 0.4869 - loss: 1.0611 - val_accuracy: 0.5525 - val_loss: 0.9911 - learning_rate: 0.0010
Epoch 3/20
806/807 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4965 - loss: 1.0476
Epoch 3: val_accuracy did not improve from 0.56946
807/807 ━━━━━━━━━━━━━━━━━━━━ 65s 80ms/step - accuracy: 0.4987 - loss: 1.0497 - val_accuracy: 0.5687 - v

#### <u><strong> Home Value Group.</strong></u>

In [ ]:
run_whole(variable="median_home_value_group")

train: 6819 images
val: 1456 images
test: 1456 images
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "EfficientNetV2S_Improved"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_images (InputLayer)       │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 12, 12, 1280)   │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,988,771 (80.07 MB)

 Trainable params: 657,411 (2.51 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


=== Feature Extraction ===
Epoch 1/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.4320 - loss: 1.1910
Epoch 1: val_accuracy improved from None to 0.54876, saving model to ../Results/best_efficientnetv2s_median_home_value_group.keras

Epoch 1: finished saving model to ../Results/best_efficientnetv2s_median_home_value_group.keras
427/427 ━━━━━━━━━━━━━━━━━━━━ 104s 158ms/step - accuracy: 0.4622 - loss: 1.1316 - val_accuracy: 0.5488 - val_loss: 1.0133 - learning_rate: 0.0010
Epoch 2/20
426/427 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5038 - loss: 1.0637
Epoch 2: val_accuracy did not improve from 0.54876
427/427 ━━━━━━━━━━━━━━━━━━━━ 34s 81ms/step - accuracy: 0.4999 - loss: 1.0623 - val_accuracy: 0.5234 - val_loss: 1.0402 - learning_rate: 0.0010
Epoch 3/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5060 - loss: 1.0543
Epoch 3: val_accuracy did not improve from 0.54876
427/427 ━━━━━━━━━━━━━━━━━━━━ 35s 81ms/step - accuracy: 0.5168 - loss: 1.0457 - val_accuracy:

<div align="left">

<h2><b><u>Model II: LSTM Multi-View.</u></b></h2>

</div>

#### <u><strong> Eduction Group.</strong></u>

In [42]:
run_LSTM(variable = "shr_college_group")

Total locations with all 4 headings: 3481 (13924 images)
Train: 2436 | Val: 522 | Test: 523 locations
Steps — train: 153 | val: 33 | test: 33
Image batch shape : (16, 4, 384, 384, 3)
Label batch shape : (16,)


Model: "EfficientNetV2S_LSTM_MultiView"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ multiview_input (InputLayer)    │ (None, 4, 384, 384, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_augmentation                 │ (None, 4, 384, 384, 3) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_backbone (TimeDistributed)   │ (None, 4, 12, 12,      │    20,331,360 │
│                                 │ 1280)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_gap (TimeDistributed)        │ (None, 4, 1280)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ view_lstm (LSTM)                │ (None, 256)            │     1,573,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,038,371 (84.07 MB)

 Trainable params: 1,707,011 (6.51 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


=== Phase 1: Feature Extraction (frozen backbone) ===
Epoch 1/20
153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.3998 - loss: 1.1434
Epoch 1: val_accuracy improved from None to 0.39847, saving model to best_efficientnetv2s_lstm.keras

Epoch 1: finished saving model to best_efficientnetv2s_lstm.keras
153/153 ━━━━━━━━━━━━━━━━━━━━ 141s 362ms/step - accuracy: 0.4056 - loss: 1.1207 - val_accuracy: 0.3985 - val_loss: 1.1070 - learning_rate: 0.0010
Epoch 2/20
153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - accuracy: 0.4774 - loss: 1.0524
Epoch 2: val_accuracy improved from 0.39847 to 0.50192, saving model to best_efficientnetv2s_lstm.keras

Epoch 2: finished saving model to best_efficientnetv2s_lstm.keras
153/153 ━━━━━━━━━━━━━━━━━━━━ 33s 218ms/step - accuracy: 0.4770 - loss: 1.0511 - val_accuracy: 0.5019 - val_loss: 1.0362 - learning_rate: 0.0010
Epoch 3/20
153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - accuracy: 0.4976 - loss: 1.0242
Epoch 3: val_accuracy improved from 0.50192 to 0.5229

#### <u><strong> Vacancy Group.</strong></u>

In [43]:
run_LSTM(variable = "vacancy_rate_group")

Total locations with all 4 headings: 4606 (18424 images)
Train: 3224 | Val: 691 | Test: 691 locations
Steps — train: 202 | val: 44 | test: 44
Image batch shape : (16, 4, 384, 384, 3)
Label batch shape : (16,)


Model: "EfficientNetV2S_LSTM_MultiView"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ multiview_input (InputLayer)    │ (None, 4, 384, 384, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_augmentation                 │ (None, 4, 384, 384, 3) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_backbone (TimeDistributed)   │ (None, 4, 12, 12,      │    20,331,360 │
│                                 │ 1280)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_gap (TimeDistributed)        │ (None, 4, 1280)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ view_lstm (LSTM)                │ (None, 256)            │     1,573,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,038,371 (84.07 MB)

 Trainable params: 1,707,011 (6.51 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


=== Phase 1: Feature Extraction (frozen backbone) ===
Epoch 1/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.4118 - loss: 1.1033
Epoch 1: val_accuracy improved from None to 0.50796, saving model to best_efficientnetv2s_lstm.keras

Epoch 1: finished saving model to best_efficientnetv2s_lstm.keras
202/202 ━━━━━━━━━━━━━━━━━━━━ 154s 327ms/step - accuracy: 0.4662 - loss: 1.0492 - val_accuracy: 0.5080 - val_loss: 1.0076 - learning_rate: 0.0010
Epoch 2/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.5070 - loss: 1.0129
Epoch 2: val_accuracy improved from 0.50796 to 0.52243, saving model to best_efficientnetv2s_lstm.keras

Epoch 2: finished saving model to best_efficientnetv2s_lstm.keras
202/202 ━━━━━━━━━━━━━━━━━━━━ 45s 221ms/step - accuracy: 0.4997 - loss: 0.9991 - val_accuracy: 0.5224 - val_loss: 0.9826 - learning_rate: 0.0010
Epoch 3/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.5127 - loss: 0.9933
Epoch 3: val_accuracy improved from 0.52243 to 0.5369

#### <u><strong> Home Value Group.</strong></u>

In [44]:
run_LSTM(variable = "median_home_value_group")

Total locations with all 4 headings: 2432 (9728 images)
Train: 1702 | Val: 365 | Test: 365 locations
Steps — train: 107 | val: 23 | test: 23
Image batch shape : (16, 4, 384, 384, 3)
Label batch shape : (16,)


Model: "EfficientNetV2S_LSTM_MultiView"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ multiview_input (InputLayer)    │ (None, 4, 384, 384, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_augmentation                 │ (None, 4, 384, 384, 3) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_backbone (TimeDistributed)   │ (None, 4, 12, 12,      │    20,331,360 │
│                                 │ 1280)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_gap (TimeDistributed)        │ (None, 4, 1280)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ view_lstm (LSTM)                │ (None, 256)            │     1,573,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,038,371 (84.07 MB)

 Trainable params: 1,707,011 (6.51 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


=== Phase 1: Feature Extraction (frozen backbone) ===
Epoch 1/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.4325 - loss: 1.1034
Epoch 1: val_accuracy improved from None to 0.50411, saving model to best_efficientnetv2s_lstm.keras

Epoch 1: finished saving model to best_efficientnetv2s_lstm.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 134s 432ms/step - accuracy: 0.4759 - loss: 1.0720 - val_accuracy: 0.5041 - val_loss: 1.0174 - learning_rate: 0.0010
Epoch 2/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - accuracy: 0.5250 - loss: 0.9947
Epoch 2: val_accuracy improved from 0.50411 to 0.51781, saving model to best_efficientnetv2s_lstm.keras

Epoch 2: finished saving model to best_efficientnetv2s_lstm.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 25s 235ms/step - accuracy: 0.5353 - loss: 0.9795 - val_accuracy: 0.5178 - val_loss: 0.9939 - learning_rate: 0.0010
Epoch 3/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.5513 - loss: 0.9558
Epoch 3: val_accuracy improved from 0.51781 to 0.5397